# NanoGPT (Learn)

In [1]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [2]:
import os
import sys
from pathlib import Path

from pathlib import Path

CWD = os.path.realpath(os.getcwd())
PARENT_DIR = os.path.dirname(CWD)
sys.path.append(PARENT_DIR)

DATA_DIR = Path(PARENT_DIR).parent / 'data'

In [3]:
from reader.loader import TextDataset

dataset = TextDataset(DATA_DIR, device=device)

100%|██████████| 5/5 [00:00<00:00, 12024.95it/s]


In [4]:
from src.modules.architecture.ngram_lm import NgramLanguageModel
from reader.preprocess import decode
import torch

BATCH_SIZE = 16
EMBEDDING_SIZE = 64
SEQ_LENGTH = 32
DROPOUT_RATE = 0.2

N_HEADS = 8
N_GROUPS = 4
N_BLOCKS = 4
LR = 1e-3

EVAL_ITER = 100
EVAL_INTERVAL = 100
EPOCH_SIZE = 10000

torch.manual_seed(1337)

model = NgramLanguageModel(vocab_size=dataset.vocab_size, 
                           n_heads=N_HEADS, 
                           n_groups=N_GROUPS,
                           embedding_size=EMBEDDING_SIZE, 
                           seq_length=SEQ_LENGTH, 
                           n_blocks=N_BLOCKS, 
                           dropout_rate=DROPOUT_RATE,
                           device=device,
                           attention_type='causal',
                           multi_attention_type='gqa',
                           position_embedding_type='rope',
                           enable_kv_cache=True)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

print("Number of parameters:")
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

Number of parameters:
0.194393 M parameters


In [5]:
@torch.no_grad()
def estimate_loss(x, y, model):
    losses = torch.zeros(EVAL_ITER)
    for k in range(EVAL_ITER):
        logits, loss = model(x, y)
        losses[k] = loss.item()
    return losses.mean()

In [6]:
for iter in range(EPOCH_SIZE):

    # every once in a while evaluate the loss on train and val sets
    if iter % EVAL_INTERVAL == 0 or iter == EPOCH_SIZE - 1:
        model.eval()
        x_train, y_train = dataset.load_train(batch_size=BATCH_SIZE, context_window_size=SEQ_LENGTH)
        x_val, y_val = dataset.load_test(batch_size=BATCH_SIZE, context_window_size=SEQ_LENGTH)
        train_losses = estimate_loss(x_train, y_train, model)
        val_losses = estimate_loss(x_val, y_val, model)
        print(f"step {iter}: train loss {train_losses:.4f}, val loss {val_losses:.4f}")
        model.train()

    # sample a batch of data
    x_train, y_train = dataset.load_train(BATCH_SIZE)

    # evaluate the loss
    logits, loss = model(x_train, y_train)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 4.4757, val loss 4.4775
step 100: train loss 2.4235, val loss 2.4060
step 200: train loss 2.1100, val loss 2.1872
step 300: train loss 1.9749, val loss 2.0423
step 400: train loss 1.8711, val loss 1.9498
step 500: train loss 1.8190, val loss 1.9298
step 600: train loss 1.7650, val loss 1.8860
step 700: train loss 1.7199, val loss 1.8083
step 800: train loss 1.6325, val loss 1.8174
step 900: train loss 1.6334, val loss 1.7676
step 1000: train loss 1.6308, val loss 1.7245
step 1100: train loss 1.5808, val loss 1.7640
step 1200: train loss 1.5708, val loss 1.7797
step 1300: train loss 1.5585, val loss 1.6596
step 1400: train loss 1.5433, val loss 1.7447
step 1500: train loss 1.6100, val loss 1.6899
step 1600: train loss 1.4964, val loss 1.7277
step 1700: train loss 1.5171, val loss 1.7470
step 1800: train loss 1.5660, val loss 1.6857
step 1900: train loss 1.5141, val loss 1.6680
step 2000: train loss 1.4766, val loss 1.6816
step 2100: train loss 1.5017, val loss 1.6246


In [7]:
# generate from the model
model.eval()
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(dataset.itos, model.generate(context, max_new_tokens=2000)[0].tolist()))


pe condenter with did fact your my study was brother payment to promission, why haste is, like him, generords in holoughld in the word. Yet he liss persible in the was exists."

What married, his, family, who in child of the spoint I may case frequre known and immand, not declived in spite of his province, but were I know showed, a seculous be a care.

LAPAULIO:
And constand also let in the elder who was a fact two so.

He had barried any the House, the slong Kelteroved Both's turned the crazy was a gless wife. Grigory, a place. Thinkiss, (noten he were in the word 'made at heasted amily's poor thy keep and runaway a place of the ordered not of your doubt, turned me, though he used to it was have which because but impations, Mitya, the though heard to kept into other. Three married, that it is  that  or  Difies in  there,  if  exclaming  or  property  the  and  imagine  type  of  mean  his  socow.

DUKE  ELINS:
That years to the kept to the creatimes for imagination of her in succepin

In [8]:
from reader.preprocess import encode
context = torch.tensor([encode(dataset.stoi, 'fyodor')], dtype=torch.long, device=device)
print(decode(dataset.itos, model.generate(context, max_new_tokens=2000)[0].tolist()))

fyodor Pavlovitch. At is a man that the house, and I was none,
I know sure embleris servect for the doner of they was of the Moscow, I begin woman acquaintance believed by the began up to the qui Alyosha was care (2848, hitbe the hope, clitch your sove for it.

This was part they had no began (8) very family of the pospecially say- Oubt, and was place of Mitya. He did no speak. Meany man not last for a braken to soons in they hais death sixtcus early was believe and an extropt.
But she though new its, from his sense; in their by the property with a stumpered's debaKENTIGA:
And your such must instance have been viery began of enterely each began that he was a family others of Nour consistened that this persons for standing and that he used loved in his father constage possant of the strange, and generate of our be not, pray fell the could not funner to satisfied fact was all described with the wrise after between reson of our will with delication and they straight they way the life mean

In [9]:
context = torch.tensor([encode(dataset.stoi, 'slab')], dtype=torch.long, device=device)
print(decode(dataset.itos, model.generate(context, max_new_tokens=2000)[0].tolist()))

slab"  Pavlovitch Karamazovan's grews signifies confessed not? I not my desire, Dmitri: me your
Delaïda Ivanovna, Land King Voal merely grieva's get rights of the contrast understanding property, and an expossible".  And it  senseless:  for many  like  the  say-temped  which  one  for in  a  sentences.   I  character,  be  in  itexpressed  in  be.

POMPHNo):  one.—And  I   a  married which  will  not  sourr  sentenced  the  hide  their  a sultened  if  it  chall  it  sense.—I  of  the  kind."  (Every had  "that  was treighness  recan  and  gener!

PEDNCE:
Begarder by your reast, there to important to his down, and my of it; of the Duke of the coaprarided age of word-have him: fit more? When, they was Jelipt, and it always, if a plabily trunk.

Whom he money
Grigory, had been they general into money, and caried, keeping. In the lefterent sult with all hands was gravely was let on his lcose children.

MERICUS:
Lard was buried afterwards cripted, who grew a Garriage.At Eshowed that Fyodor